# From called sets to programs, with a planted truth (notebook 102)

**The question.** Notebook 100 found that a planted pathway difference spills into the sets that share its genes,
so the number of called sets is not the number of findings. Which grouping of the positional F-test's calls —
notebook 71's redundancy clusters (RED), notebook 75's overlap programs (OVL) or notebook 93's driver-gene networks
(NET) — turns them back into the planted programs, one per planted set, with the fewest programs that hold no planted
set?

**The frozen protocol** is `docs/results/pt-planted-programs.md` (commit 81b1dd0). Notebook 100's planted data are
rebuilt with its seeds (identical); only the positional F-test is run, keeping per-gene z and q.

**Sections.** 1 setup, inputs and guards; 2 the F-test baseline; 3 planted replicates; 4 groupings and metrics;
5 readings; 6 figures; 7 reading and run record.

## 1 · Setup, inputs and guards

In [ ]:
import argparse
import json
import os
import sys
import time
import warnings
from dataclasses import replace
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
package_root = results_root / 'package_pseudospace'
T71 = results_root / 'pt_pathways_package_pseudospace'
output = results_root / 'pt_planted_programs'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '102.planted_programs.1'
SEED = 100
DS = 'aki_vs_control'
FINGERPRINTS = {'hvm': 'efafc9ac2e99', 'aki': 'bb4a80c1b484'}
BASE, NULL_LABEL = 'swap_IR2A2_Ctrl1A4', 'swap_IR2A2_Ctrl1A2'       # file and cache slugs
BASE_P, NULL_P = 'swap:IR2A2+Ctrl1A4', 'swap:IR2A2+Ctrl1A2'           # partition labels
LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
ALPHA, CAMERA_COR, DF1, DF2 = .05, .01, 6, 6
SET_SIZES = (10, 300)
K_SETS, PLANT_SIZES, MAX_JACCARD, FRAC, MIN_PLANTED = 15, (20, 150), .1, .6, 10
SCENARIOS, AMPLITUDES, N_REP = ('shape', 'located', 'level', 'bump'), (.5, 1.), 8
DETECT_SENS, DETECT_FALSE, TIE = .5, 2., .1
ZONE_MARKERS = {'PT-S1': ['Slc5a12'], 'PT-S2': ['Slc13a3'], 'PT-S3': ['Slc16a9']}
ZONE_SHORT = {'PT-S1': 'S1', 'PT-S2': 'S2', 'PT-S3': 'S3'}
NODE_SPACING = 1 / 29
EXPRESSION_COLUMN = 'expression_per_10k'
METHODS = ['positional F-test', 'segment F-test', 'clustering']
REQUIRED = [package_root / 'hvm' / 'pt_units.h5ad', package_root / 'aki' / 'pt_units.h5ad',
            results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv',
            T71 / 'tables' / f'gene_statistics_{DS}.csv', T71 / 'tables' / f'gene_sets_{DS}.json',
            T71 / 'tables' / f'universe_{DS}.csv', T71 / 'tables' / 'method_b_zones.csv',
            T71 / 'tables' / f'pathway_tests_{DS}_cameraPR_{BASE}.csv']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70 and 71 first)')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from pseudospace.discrete_comparators import segment_step_lr
from pseudospace.package_inputs import aki_inputs, hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import (GeneModel, camera_pr, clustering_first_inputs, moderated_f, partition_groups,
                                           single_split_design, specimen_shape_lr, standard_gene_model)
from pseudospace.planting import choose_sets, effect_shape, plant_counts
from pseudospace.stage_cache import cached_payload, code_digest

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
METHOD_COLOR = {'positional F-test': '#1b8a5a', 'segment F-test': '#5fb88a', 'clustering': '#eda100'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []
orth = pd.read_csv(results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv')
MT = sorted(g for g in orth.mouse_symbol.astype(str) if g.startswith('mt-'))
U = {k: read_pt_units(package_root / k / 'pt_units.h5ad', k) for k in ('hvm', 'aki')}
for k, u in U.items():
    assert u.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINTS[k], k
kept_h = hvm_inputs(U['hvm'], data_root, orth)
D = aki_inputs(U['aki'], data_root, orth, kept_h['genes'], exclude_genes=MT)
del kept_h, U
GENES = list(D['genes'])
SETS = D['gene_sets']
assert GENES == pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv').gene.astype(str).tolist(), 'G1: universe'
assert SETS == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G1: sets'
COUNTS = D['counts'].tocsr()
POS, SPEC, LIB = np.asarray(D['position'], float), np.asarray(D['specimen']).astype(str), np.asarray(D['library'], float)
SEG = np.asarray(D['segment']).astype(str)
PARTS = partition_groups(SPEC, np.asarray(D['case'], bool))
assert list(PARTS)[1:] == [NULL_P, BASE_P], list(PARTS)
GROUP, NUIS = PARTS[BASE_P]
GROUP1 = sorted(set(SPEC[GROUP == 1]))
assert GROUP1 == ['Ctrl1A4', 'IR2A2'], GROUP1
LOG_MEAN = np.log(np.asarray(COUNTS.mean(axis=0)).ravel())
guard_log.append(f"G1: universe ({len(GENES):,} genes) and sets ({len(SETS):,}) equal notebook 71's; base partition "
                 f"{BASE} (group 1 = {', '.join(GROUP1)}; nuisance = the condition)")
print('\n'.join('· ' + g for g in guard_log))

## 2 · The F-test baseline

Notebook 71's cached gene model and statistics for the base partition; `moderated_f` and `camera_pr` must reproduce
71's z and calls (G2).

In [ ]:
def stored(folder, stage):
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def model_from(payload, genes):
    index = pd.Index(genes, name='gene')
    stats = pd.DataFrame({c: payload[f'stat|{c}'] for c in ('T_level', 'T_spatial', 'T_total', 'alpha', 'converged',
                                                            'separated')}, index=index)
    stats[['converged', 'separated']] = stats[['converged', 'separated']].astype(bool)
    names = sorted(k.split('|', 1)[1] for k in payload if k.startswith('curve|'))
    return GeneModel(statistics=stats, grid=payload['grid'], knots=payload['knots'], reference=payload['reference'],
                     case=payload['case'], delta=payload['delta'], se=payload['se'],
                     specimen_curves={n: payload[f'curve|{n}'] for n in names},
                     specimen_converged=pd.DataFrame({n: payload[f'converged|{n}'].astype(bool) for n in names}, index=index),
                     specimen_separated=pd.DataFrame({n: payload[f'separated|{n}'].astype(bool) for n in names}, index=index),
                     groups=pd.Series(payload['group_values'].astype(int), index=payload['group_names'].astype(str)),
                     dispersion={'alpha': payload['alpha'], 'converged': payload['alpha_converged'].astype(bool)})


BASE_MODEL = model_from(stored(T71, f'gene_model_{DS}_{BASE}'), GENES)
table71 = pd.read_csv(T71 / 'tables' / f'gene_statistics_{DS}.csv', header=[0, 1], index_col=0)['relabeling 2']
assert np.allclose(table71.T_spatial.to_numpy(), BASE_MODEL.statistics.T_spatial.to_numpy(), rtol=1e-12, atol=0)
BASE_T, BASE_LR = table71.T_spatial.to_numpy(float), table71.LR_spec.to_numpy(float)
ALPHA_ALL = np.asarray(BASE_MODEL.dispersion['alpha'], float)


def f_calls(T, LR):
    z = moderated_f(pd.Series(T, index=GENES), DF1, LR, DF2, covariate=LOG_MEAN, robust=False).z
    return set(camera_pr(z, SETS, inter_gene_cor=CAMERA_COR).query('call').pathway_id), z


BASE_CALLS = {}
BASE_CALLS['positional F-test'], z0 = f_calls(BASE_T, BASE_LR)
np.testing.assert_allclose(z0.to_numpy(), table71.z.to_numpy(), rtol=1e-9, atol=1e-9)
cam71 = pd.read_csv(T71 / 'tables' / f'pathway_tests_{DS}_cameraPR_{BASE}.csv')
assert BASE_CALLS['positional F-test'] == set(cam71.pathway_id[cam71.call.astype(bool)]), 'G2: F-test calls'
guard_log.append(f"G2: the cached model reproduces 71's z and its {len(BASE_CALLS['positional F-test'])} F-test calls on the base partition")

## 3 · Planted replicates (notebook 100's planting, positional F-test only)

In [ ]:
from pseudospace.pathway_decoys import overlap_programs
from pseudospace.pathway_networks import driver_jaccard, leiden_programs
from pseudospace.stats_gam import bh_adjust

GIDX = pd.Series(np.arange(len(GENES)), index=GENES)
G1_UNITS = GROUP == 1
K_LOW = 3
TOP_SHARE, MIN_DRIVERS, PROGRAM_CUT = .10, 3, .5
JACCARD_EDGE, RESOLUTION, LEIDEN_SEED = .1, 1., 1234
USABLE0 = (BASE_MODEL.statistics.converged & ~BASE_MODEL.statistics.separated).to_numpy()


def f_frame(T, LR):
    m = moderated_f(pd.Series(T, index=GENES), DF1, LR, DF2, covariate=LOG_MEAN, robust=False)
    m['q'] = bh_adjust(m.p.to_numpy())
    return m


def planted_f(scenario, amplitude, rep, k=K_SETS, arm=0):
    """Notebook 100's run_replicate up to the F-test (same random stream), returning z, q, usable and the calls."""
    seed = [SEED, SCENARIOS.index(scenario), int(amplitude * 10), rep] + ([arm] if arm else [])
    rng = np.random.default_rng(seed)
    chosen = choose_sets(SETS, rng, k=k, sizes=PLANT_SIZES, max_jaccard=MAX_JACCARD, frac=FRAC, min_members=MIN_PLANTED)
    signs = {s: float(rng.choice([-1., 1.])) for s in chosen}
    genes = [g for s in chosen for g in chosen[s]]
    gsign = np.array([signs[s] for s in chosen for _ in chosen[s]])
    idx = GIDX[genes].to_numpy()
    f = effect_shape(POS, scenario, G1_UNITS)
    mult = np.ones((len(POS), len(idx)))
    mult[G1_UNITS] = np.exp(np.log(2) * amplitude * f[G1_UNITS, None] * gsign[None, :])
    counts = plant_counts(COUNTS[:, idx].toarray(), mult, rng)
    alpha = ALPHA_ALL[idx]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        refit = standard_gene_model(counts, LIB, POS, GROUP, SPEC, grid=BASE_MODEL.grid, genes=genes, nuisance_shape=NUIS,
                                    dispersion=alpha)
        spec = specimen_shape_lr(counts, LIB, POS, GROUP, SPEC, {'alpha': alpha, 'converged': np.ones(len(idx), bool)},
                                 nuisance_shape=NUIS)
    T, LR = BASE_T.copy(), BASE_LR.copy()
    T[idx], LR[idx] = refit.statistics.T_spatial.to_numpy(float), spec['LR_spec']
    usable = USABLE0.copy()
    usable[idx] = (refit.statistics.converged & ~refit.statistics.separated).to_numpy()
    m = f_frame(T, LR)
    calls = set(camera_pr(m.z, SETS, inter_gene_cor=CAMERA_COR).query('call').pathway_id)
    return {'sets': np.array(list(chosen)), 'genes': np.array(genes), 'z': m.z.to_numpy(float), 'q': m.q.to_numpy(float),
            'usable': usable, 'calls': np.array(sorted(calls), dtype=str)}


CODE_KEY = code_digest(planted_f, choose_sets, effect_shape, plant_counts)
t0 = time.perf_counter()
REPL = {}
for arm, k, tag in ((0, K_SETS, 'frozen (15 sets)'), (1, K_LOW, 'low dose (3 sets)')):
    for scenario in SCENARIOS:
        for amplitude in AMPLITUDES:
            for rep in range(N_REP):
                REPL[(tag, scenario, amplitude, rep)] = cached_payload(
                    f'f_{arm}_{scenario}_{amplitude}_{rep}', lambda s=scenario, a=amplitude, r=rep, k=k, arm=arm: planted_f(s, a, r, k=k, arm=arm),
                    root=cache, params={'logic': NOTEBOOK_LOGIC_VERSION, 'key': [str(arm), scenario, str(amplitude), str(rep)]},
                    inputs={'genes': len(GENES), 'sets': len(SETS)}, verbose=False, code=CODE_KEY)
            print(f'  {tag} · {scenario} A={amplitude}: done ({time.perf_counter() - t0:.0f} s)')
# guard: the frozen arm's calls equal notebook 100's (same planting, same F-test)
B100 = results_root / 'pt_planted_benchmark' / 'stage_cache'
checked = 0
for (tag, scenario, amplitude, rep), r in REPL.items():
    if tag.startswith('frozen') and rep == 0:
        files = sorted(B100.glob(f'replicate_{scenario}_{amplitude}_{rep}__*.npz'),
                       key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
        if files:
            with np.load(files[-1], allow_pickle=False) as p100:
                assert set(p100['calls|positional F-test'].astype(str)) == set(r['calls'].astype(str)), f'G4: {scenario} {amplitude}'
                assert list(p100['sets'].astype(str)) == list(r['sets'].astype(str))
            checked += 1
guard_log.append(f"G4: the rebuilt F-test calls equal notebook 100's on {checked} checked replicates")
print('· ' + guard_log[-1])

## 4 · Groupings and metrics

In [ ]:
red = pd.read_csv(T71 / 'tables' / f'redundancy_clusters_{DS}.csv').set_index('pathway_id').representative
GROUPINGS = ['NONE', 'RED', 'OVL', 'NET']


def drivers_for(calls, z, q, usable):
    zs = pd.Series(z, index=GENES)
    top = set(zs.nlargest(int(round(TOP_SHARE * len(zs)))).index)
    use = set(np.asarray(GENES)[usable])
    sig = use & set(np.asarray(GENES)[q <= ALPHA])
    out = {}
    for c in calls:
        members = set(SETS[c])
        chosen = members & sig
        if len(chosen) < MIN_DRIVERS:
            chosen = members & use & top
        if len(chosen) < MIN_DRIVERS:
            chosen = members & use
        out[c] = chosen
    return out


def group(calls, z, q, usable):
    calls = sorted(calls)
    if not calls:
        return {g: pd.Series(dtype=int) for g in GROUPINGS}
    zs = pd.Series(z, index=GENES)
    out = {'NONE': pd.Series(np.arange(1, len(calls) + 1), index=calls)}
    out['RED'] = pd.Series(pd.factorize(red.reindex(calls).to_numpy())[0] + 1, index=calls)
    contributing = set(zs.nlargest(int(round(TOP_SHARE * len(zs)))).index)
    out['OVL'] = overlap_programs(SETS, calls, contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id').program.reindex(calls)
    names, J = driver_jaccard(drivers_for(calls, z, q, usable))
    out['NET'] = leiden_programs(names, J, threshold=JACCARD_EDGE, resolution=RESOLUTION, seed=LEIDEN_SEED).reindex(calls)
    return out


def metrics(prog, planted):
    if prog.empty:
        return {'programs': 0, 'true programs': 0, 'false programs': 0, 'separation': np.nan, 'containment': np.nan}
    has = prog.groupby(prog).apply(lambda s: bool(set(s.index) & planted))
    called_planted = [p for p in planted if p in prog.index]
    true_ids = set(has.index[has])
    return {'programs': int(prog.nunique()), 'true programs': int(has.sum()), 'false programs': int((~has).sum()),
            'separation': len({prog[p] for p in called_planted}) / len(called_planted) if called_planted else np.nan,
            'containment': float(prog.isin(true_ids).mean())}


t0 = time.perf_counter()
ROWS = []
for (tag, scenario, amplitude, rep), r in REPL.items():
    planted = set(r['sets'].astype(str))
    groups = group(set(r['calls'].astype(str)), r['z'], r['q'], r['usable'].astype(bool))
    for g in GROUPINGS:
        ROWS.append({'arm': tag, 'scenario': scenario, 'A': amplitude, 'replicate': rep, 'grouping': g, 'calls': len(r['calls']),
                     **metrics(groups[g], planted)})
REP = pd.DataFrame(ROWS)
REP.to_csv(tables / 'replicate_programs.csv', index=False)
m0 = f_frame(BASE_T, BASE_LR)
base_groups = group(BASE_CALLS['positional F-test'], m0.z.to_numpy(), m0.q.to_numpy(), USABLE0)
BASE_PROG = pd.DataFrame([{'grouping': g, 'calls': len(BASE_CALLS['positional F-test']), 'programs': int(base_groups[g].nunique())}
                          for g in GROUPINGS])
BASE_PROG.to_csv(tables / 'baseline_programs.csv', index=False)
print(f'Groupings {time.perf_counter() - t0:.0f} s')
display(BASE_PROG)
SUMMARY = REP.groupby(['arm', 'scenario', 'A', 'grouping'])[['calls', 'programs', 'true programs', 'false programs', 'separation', 'containment']].mean().reset_index()
SUMMARY.to_csv(tables / 'summary.csv', index=False)
display(SUMMARY.pivot_table(index=['arm', 'scenario', 'A'], columns='grouping', values=['programs', 'false programs', 'separation']).round(2))

## 5 · Readings (fixed in the protocol)

In [ ]:
READ = []
for (arm, scenario, a), b in SUMMARY.groupby(['arm', 'scenario', 'A']):
    b = b[b.grouping.ne('NONE')].set_index('grouping')
    sep_ok = b[b.separation >= .8]
    if len(sep_ok):
        fewest = sep_ok['false programs'].min()
        best = sorted(sep_ok.index[sep_ok['false programs'] <= fewest + 1])
        note = ''
    else:
        best = [b.separation.idxmax()] if b.separation.notna().any() else []
        note = 'no grouping separates'
    READ.append({'arm': arm, 'scenario': scenario, 'A': a, 'best': ', '.join(best), 'note': note,
                 **{f'{g} false / separation': f"{b.loc[g, 'false programs']:.1f} / {b.loc[g, 'separation']:.2f}" for g in ('RED', 'OVL', 'NET')}})
READ = pd.DataFrame(READ)
READ.to_csv(tables / 'readings.csv', index=False)
display(READ)

## 6 · Figures

In [ ]:
GCOL = {'NONE': LIGHT, 'RED': '#8a8985', 'OVL': '#1b8a5a', 'NET': '#2a78d6'}
fig, axes = plt.subplots(2, 2, figsize=(140 * MM, 100 * MM), sharex=True)
for i, arm in enumerate(['frozen (15 sets)', 'low dose (3 sets)']):
    for j, a in enumerate(AMPLITUDES):
        ax = axes[i, j]
        for g in GROUPINGS:
            b = SUMMARY[SUMMARY.arm.eq(arm) & SUMMARY.A.eq(a) & SUMMARY.grouping.eq(g)]
            ax.scatter(b['separation'], b['false programs'], color=GCOL[g], s=14, label=g)
        ax.set_yscale('symlog')
        ax.set_title(f'{arm}, A = {a}')
        ax.set_ylabel('false programs')
for ax in axes[-1]:
    ax.set_xlabel('separation of planted sets')
axes[0, 0].legend(loc='upper left')
save(fig, '102a_false_vs_separation')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(140 * MM, 48 * MM))
for ax, arm, k in zip(axes, ['frozen (15 sets)', 'low dose (3 sets)'], (K_SETS, K_LOW)):
    b = REP[REP.arm.eq(arm) & REP.A.eq(1.)]
    data = [b[b.grouping.eq(g)].programs for g in GROUPINGS]
    ax.boxplot(data, tick_labels=GROUPINGS, showfliers=False)
    ax.axhline(k, color=MUTED, lw=.5, ls=':')
    ax.set_yscale('log')
    ax.set_title(f'{arm}, A = 1')
    ax.set_ylabel('programs per replicate')
save(fig, '102b_programs_per_replicate')
plt.show()

## 7 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', "Planting and the F-test are notebook 100's code with its seeds; G4 checks the rebuilt calls against notebook 100's cache."),
    ('reading', "Driver usability: the cached model's converged and not separated flags, replaced by the refit's flags for planted genes."),
    ('reading', "RED uses notebook 71's representative per set as the program label."),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '102_pt_planted_programs', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'replicates': len(REPL), 'baseline programs': BASE_PROG.set_index('grouping').programs.to_dict(),
       'readings': READ.to_dict('records'), 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps(RUN, indent=2, default=str))